# Training and preparing the models

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models
import torch.optim as optim  # Added missing import
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt  # For visualization
import torchvision




## Configuration


In [2]:

# Configuration
MODEL_NAME = 'vgg16'  # Options: 'inception_v3', 'vgg16', 'resnet50'
DATASET_NAME = 'cifar10'      # Options: 'cifar10', 'imagenet'
# Determine image size based on model
IMAGE_SIZE = 224


## function for loading,dataset, training, and adversarial attack

In [3]:
# function for model
def get_model(model_name, num_classes, pretrained=True, aux_logits=True):
    """
    Returns the specified pre-trained model from torchvision.

    Args:
        model_name (str): Name of the model ('resnet50', 'inception_v3', 'vgg16').
        num_classes (int): Number of classes in the dataset.
        pretrained (bool): Whether to use pre-trained weights. Default is True.

    Returns:
        torch.nn.Module: The specified model.
    """
    model_name = model_name.lower()  # Ensure the model name is case-insensitive
    if model_name == 'resnet50':
        model = models.resnet50(pretrained=pretrained)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'inception_v3':
        model = models.inception_v3(pretrained=pretrained, aux_logits=aux_logits)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'vgg16':
        model = models.vgg16(pretrained=pretrained)
        model.classifier[6] = nn.Linear(4096, num_classes)
    else:
        raise ValueError(f"Unsupported model name '{model_name}'. Choose from 'resnet50', 'inception_v3', or 'vgg16'.")

    return model

# function for dataset
# Function for dataset
def get_dataset(dataset_name, image_size=224):
    """
    Load the specified dataset with separate transformations for training and testing.

    Args:
        dataset_name (str): The name of the dataset ('cifar10' or 'imagenet').
        image_size (int): The size to which images will be resized.

    Returns:
        DataLoader: Train and test loaders for the dataset.
        int: Number of classes in the dataset.
    """
    if dataset_name == 'cifar10':
        # Training transforms with data augmentation
        train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(32, padding=4),
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), 
                                 (0.2023, 0.1994, 0.2010)),
        ])

        # Testing transforms without data augmentation
        test_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), 
                                 (0.2023, 0.1994, 0.2010)),
        ])

        train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
        test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

        num_classes = 10

    elif dataset_name == 'imagenet':
        # Training transforms with data augmentation
        train_transform = transforms.Compose([
            transforms.RandomResizedCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

        # Testing transforms without data augmentation
        test_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

        # Replace this with actual ImageNet data paths
        train_dataset = datasets.ImageFolder(root='./data/imagenet/train', transform=train_transform)
        test_dataset = datasets.ImageFolder(root='./data/imagenet/val', transform=test_transform)

        num_classes = 1000

    else:
        raise ValueError("Unsupported dataset. Choose from 'cifar10' or 'imagenet'.")

    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=4)

    return train_loader, test_loader, num_classes


def load_trained_model(model_name, dataset_name, num_classes, device):
    """
    Loads the trained model with saved weights.

    Args:
        model_name (str): The name of the model ('resnet50', 'inception_v3', 'vgg16').
        dataset_name (str): The dataset name ('cifar10' or 'imagenet').
        num_classes (int): Number of classes in the dataset.
        device (torch.device): The device to load the model onto.

    Returns:
        torch.nn.Module: The trained model loaded with saved weights.
    """
    # Initialize the model with aux_logits=False for evaluation
    model = get_model(model_name, num_classes, pretrained=True, aux_logits=True).to(device)
    model_path = f"best_{dataset_name}_{model_name}_model.pth"
    print("Loading ", model_path)
    try:
        model.load_state_dict(torch.load(model_path, map_location=device))
        print(f"Loaded model weights from '{model_path}' successfully.")
    except FileNotFoundError:
        print(f"Model weights file '{model_path}' not found.")
        raise
    model.eval()  # Set the model to evaluation mode
    return model


def channel_adversarial_attack(input_tensor, attack_channels, epsilon=0.03):
    """
    Applies adversarial perturbations to specified channels within the input tensor.

    Args:
        input_tensor (torch.Tensor): The input tensor to perturb. Shape: (N, C, H, W)
        attack_channels (list of int): List of channel indices to attack.
        epsilon (float): The magnitude of the perturbation.

    Returns:
        torch.Tensor: The perturbed input tensor.
    """
    perturbed_input = input_tensor.clone()
    # Generate noise for the specified channels across all spatial dimensions
    noise = torch.randn_like(perturbed_input[:, attack_channels, :, :]) * epsilon
    # print(noise)
    perturbed_input[:, attack_channels, :, :] += noise
    # Clamp to ensure valid pixel range
    # perturbed_input[:, attack_channels, :, :] = torch.clamp(perturbed_input[:, attack_channels, :, :], 0, 1)
    return perturbed_input



## Partitioned model

In [4]:
class PartitionedVGG16(nn.Module):
    def __init__(self, full_model, device):
        super(PartitionedVGG16, self).__init__()

        # Assign device
        self.device = device

        # Partition 1: Initial Convolutional Blocks
        self.device1 = nn.Sequential(
            # Block 1
            full_model.features[0],  # Conv2d_1
            full_model.features[1],  # ReLU
            full_model.features[2],  # Conv2d_2
            full_model.features[3],  # ReLU
            full_model.features[4],  # MaxPool

            # Block 2
            full_model.features[5],  # Conv2d_3
            full_model.features[6],  # ReLU
            full_model.features[7],  # Conv2d_4
            full_model.features[8],  # ReLU
            full_model.features[9],  # MaxPool
        ).to(self.device)

        # Partition 2: Middle Convolutional Blocks
        self.device2 = nn.Sequential(
            # Block 3
            full_model.features[10],  # Conv2d_5
            full_model.features[11],  # ReLU
            full_model.features[12],  # Conv2d_6
            full_model.features[13],  # ReLU
            full_model.features[14],  # Conv2d_7
            full_model.features[15],  # ReLU
            full_model.features[16],  # MaxPool
        ).to(self.device)

        # Partition 3: Deeper Convolutional Blocks
        self.device3 = nn.Sequential(
            # Block 4
            full_model.features[17],  # Conv2d_8
            full_model.features[18],  # ReLU
            full_model.features[19],  # Conv2d_9
            full_model.features[20],  # ReLU
            full_model.features[21],  # Conv2d_10
            full_model.features[22],  # ReLU
            full_model.features[23],  # MaxPool

            # Block 5
            full_model.features[24],  # Conv2d_11
            full_model.features[25],  # ReLU
            full_model.features[26],  # Conv2d_12
            full_model.features[27],  # ReLU
            full_model.features[28],  # Conv2d_13
            full_model.features[29],  # ReLU
            full_model.features[30],  # MaxPool
        ).to(self.device)

        # Partition 4: Fully Connected Layers (Classifier)
        self.device4 = nn.Sequential(
            nn.Flatten(),
            full_model.classifier[0],  # Linear_1
            full_model.classifier[1],  # ReLU
            full_model.classifier[2],  # Dropout
            full_model.classifier[3],  # Linear_2
            full_model.classifier[4],  # ReLU
            full_model.classifier[5],  # Dropout
            full_model.classifier[6],  # Linear_3
        ).to(self.device)

    def forward(self, x, attack_specs={}):
        """
        Forward pass through the partitioned VGG16 model.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Final output logits.
        """
                # Device 1
        if 'device1' in attack_specs:
            for attack in attack_specs['device1']:
                attack_channels = attack.get('channels', list(range(x.size(1))))
                epsilon = attack.get('epsilon', 0.03)
                x = channel_adversarial_attack(x, attack_channels, epsilon)
        x = self.device1(x)

        # Device 2
        if 'device2' in attack_specs:
            for attack in attack_specs['device2']:
                attack_channels = attack.get('channels', list(range(x.size(1))))
                epsilon = attack.get('epsilon', 0.03)
                x = channel_adversarial_attack(x, attack_channels, epsilon)
        x = self.device2(x)

        # Device 3
        if 'device3' in attack_specs:
            for attack in attack_specs['device3']:
                attack_channels = attack.get('channels', list(range(x.size(1))))
                epsilon = attack.get('epsilon', 0.03)
                x = channel_adversarial_attack(x, attack_channels, epsilon)
        x = self.device3(x)

        # Device 4
        if 'device4' in attack_specs:
            for attack in attack_specs['device4']:
                attack_channels = attack.get('channels', list(range(x.size(1))))
                epsilon = attack.get('epsilon', 0.03)
                x = channel_adversarial_attack(x, attack_channels, epsilon)
        x = self.device4(x)

        return x  # Return the final output logits


## Main Execution Block

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load dataset
_, test_loader, num_classes = get_dataset(DATASET_NAME, image_size=IMAGE_SIZE)

# Load the trained model
full_model = load_trained_model(MODEL_NAME, DATASET_NAME, num_classes, device)

# Create the partitioned model based on the model type
if MODEL_NAME == 'vgg16':
    partitioned_model = PartitionedVGG16(full_model, device)      # Ensure this class is defined similarly
else:
    raise ValueError(f"Unsupported model name '{MODEL_NAME}'.")

partitioned_model.to(device)
partitioned_model.eval()

# Specify where to insert adversarial attacks

attack_specs = {
    'device2': [
        {
            'channels': [0,1,2,3,4,5,6,7,8,9],  # Channel indices start from 0
            'epsilon': 5
        },
        {
            'channels': [5,6,7,8,9],        # Attack only channel 5
            'epsilon': 0
        }
    ],
    'device3': [
        {
            'channels': [0,1,2,3,4],        # Attack only channel 0
            'epsilon': 0
        }
    ],
 
}

# Initialize counters for accuracy
total_correct_before = 0
total_correct_after = 0
total_samples = 0

# Disable gradient computation for efficiency
with torch.no_grad():
    # Iterate over the test dataset
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device)
        labels = labels.to(device)

        # ---------------------------
        # 1. Inference Without Attack
        # ---------------------------
        outputs_before = partitioned_model(images)  # No attacks applied

        final_output_before = outputs_before
        _, predicted_before = torch.max(final_output_before, 1)
        total_correct_before += (predicted_before == labels).sum().item()

        # ---------------------------
        # 2. Inference With Attack
        # ---------------------------
        outputs_after = partitioned_model(images, attack_specs=attack_specs)  # Attacks applied
        final_output_after = outputs_after
        _, predicted_after = torch.max(final_output_after, 1)
        total_correct_after += (predicted_after == labels).sum().item()

        # Update total samples
        total_samples += labels.size(0)

        # (Optional) Print progress
        if (batch_idx + 1) % 10 == 0 or (batch_idx + 1) == len(test_loader):
            print(f"Processed {total_samples} samples")

# Calculate and print final accuracies
accuracy_before = 100 * total_correct_before / total_samples
accuracy_after = 100 * total_correct_after / total_samples
print(f"Test Accuracy before adversarial attacks: {accuracy_before:.2f}%")
print(f"Test Accuracy after adversarial attacks: {accuracy_after:.2f}%")




Files already downloaded and verified
Files already downloaded and verified


c:\Users\Admin\.conda\envs\ML_AIoT\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Admin\.conda\envs\ML_AIoT\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading  best_cifar10_vgg16_model.pth
Loaded model weights from 'best_cifar10_vgg16_model.pth' successfully.
Processed 640 samples
Processed 1280 samples
Processed 1920 samples
Processed 2560 samples
Processed 3200 samples
Processed 3840 samples
Processed 4480 samples
Processed 5120 samples
Processed 5760 samples
Processed 6400 samples
Processed 7040 samples
Processed 7680 samples
Processed 8320 samples
Processed 8960 samples
Processed 9600 samples
Processed 10000 samples
Test Accuracy before adversarial attacks: 89.00%
Test Accuracy after adversarial attacks: 22.18%
